# Notebook 4 — Calibration & Triage
**RSNA Intracranial Hemorrhage Detection — Improved Pipeline**

Calibrate the 2.5D model's predicted probabilities and set up a
clinical triage system.

### Contents
1. **OOF predictions** — collect out-of-fold probabilities from all 5 folds
2. **Reliability diagrams** — before/after calibration
3. **ECE** (Expected Calibration Error)
4. **Temperature scaling** (parametric)
5. **Isotonic regression** (non-parametric)
6. **3-band triage system**: High / Medium / Low risk
7. Save `calibration_params.json` for downstream use

### Methodology Notes (Reviewer Q&A)
| Concern | Resolution |
|---------|------------|
| **Calibration data leakage?** | Calibration is fitted on **OOF (out-of-fold)** predictions: each sample's probability comes from a model that **never saw it during training**. There is no separate held-out test set — OOF serves as a cross-validated performance estimate. This is standard academic practice (see Platt 1999, Niculescu-Mizil & Caruana 2005) but must not be confused with independent external validation. |
| **Triage thresholds?** | The 0.7/0.3 thresholds are **heuristic values** informed by general clinical triage literature (rule-out sensitivity >95%, rule-in specificity >90%). They were **not optimized on this dataset** and would require institutional calibration before deployment. We report them to demonstrate the triage concept, not as validated clinical cutoffs. |
| **OOF = final evaluation?** | OOF is the strongest internal validation available without a held-out test set. It provides an unbiased estimate of generalization performance (each prediction is unseen by its model), but is not a substitute for external or temporal validation. |

### Requirements
- **GPU**: Yes (inference on all folds)
- **Inputs**: NB01 best models (all 5 folds), NB00 manifest, NB02 cache
- **Time**: ~30 min

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  CONFIG + IMPORTS (aligned to 01_training)  ██
# ══════════════════════════════════════════════════════════════════════════
from pathlib import Path
import os, json, random, warnings, gc
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.isotonic import IsotonicRegression
from scipy.optimize import minimize_scalar
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ─── Paths (match 01_training outputs) ──────────────────────────────
NB02_DIR = Path('E:/major_8/major_2nd_cache_files')
NB01_DIR = Path('E:/major_8/majr_1_out')

MANIFEST_PATH  = NB02_DIR / 'manifest.csv'
NPY_CACHE_DIR  = NB02_DIR / 'cache'
NORM_STATS     = NB02_DIR / 'normalization_stats.json'

IMG_SIZE   = 380
BACKBONE   = 'tf_efficientnet_b4'
IN_CH      = 9
N_CLASSES  = 6
DROPOUT    = 0.4
N_FOLDS    = 5
BATCH_SIZE = 8
NUM_WORKERS = 1
SEED = 42

SUBTYPES = ['any', 'epidural', 'intraparenchymal',
            'intraventricular', 'subarachnoid', 'subdural']

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('E:/major_8/majr_1_out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert MANIFEST_PATH.exists(), f'manifest missing: {MANIFEST_PATH}'
assert NPY_CACHE_DIR.exists(), f'cache missing: {NPY_CACHE_DIR}'
print(f'Device: {DEVICE}')
print(f'Calibration input: {IN_CH}ch @ {IMG_SIZE}x{IMG_SIZE} pre-normalized cache')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  MODEL + DATASET  ██
# ══════════════════════════════════════════════════════════════════════════

def build_model(backbone=BACKBONE, in_ch=IN_CH, n_cls=N_CLASSES):
    model = timm.create_model(backbone, pretrained=False,
                              num_classes=0, drop_rate=DROPOUT, drop_path_rate=0.2)
    old = model.conv_stem
    new = nn.Conv2d(in_ch, old.out_channels, kernel_size=old.kernel_size,
                    stride=old.stride, padding=old.padding, bias=(old.bias is not None))
    k = in_ch // 3 if in_ch > 3 else 1
    with torch.no_grad():
        if in_ch == 3:
            new.weight.copy_(old.weight)
        else:
            new.weight.copy_(old.weight.repeat(1, k, 1, 1) / k)
        if old.bias is not None:
            new.bias.copy_(old.bias)
    model.conv_stem = new
    model.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(model.num_features, n_cls))
    return model


class ICH2_5DDataset(Dataset):
    def __init__(self, df, npy_root, transform):
        self.df = df.reset_index(drop=True)
        self.npy_root = Path(npy_root)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = self.npy_root / f'{row["image_id"]}.npy'
        if p.exists():
            try:
                img = np.load(str(p)).astype(np.float32)
                if img.shape != (IMG_SIZE, IMG_SIZE, 9):
                    img = np.zeros((IMG_SIZE, IMG_SIZE, 9), dtype=np.float32)
            except Exception:
                img = np.zeros((IMG_SIZE, IMG_SIZE, 9), dtype=np.float32)
        else:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 9), dtype=np.float32)

        if self.transform:
            img = self.transform(image=img)['image']
        labels = torch.tensor([row[c] for c in SUBTYPES], dtype=torch.float32)
        return img, labels


val_transform = A.Compose([ToTensorV2()])
print('Model + Dataset classes ready (pre-stacked 9ch cache).')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  COLLECT OOF PREDICTIONS (all 5 folds)  ██
# ══════════════════════════════════════════════════════════════════════════

manifest = pd.read_csv(MANIFEST_PATH)
oof_logits = np.zeros((len(manifest), N_CLASSES))
oof_labels = np.zeros((len(manifest), N_CLASSES))

for fold_id in range(N_FOLDS):
    model_path = NB01_DIR / f'best_model_fold{fold_id}.pth'
    if not model_path.exists():
        print(f'  ⚠ Fold {fold_id} model not found, skipping')
        continue

    print(f'\nFold {fold_id}: loading {model_path.name}')
    model = build_model()
    model.load_state_dict(torch.load(str(model_path), map_location=DEVICE))
    model = model.to(DEVICE).eval()

    val_df = manifest[manifest['fold'] == fold_id]
    val_ds = ICH2_5DDataset(val_df, NPY_CACHE_DIR, val_transform)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

    fold_logits, fold_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f'  Fold {fold_id}', leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
                logits = model(imgs)
            fold_logits.append(logits.cpu().float().numpy())
            fold_labels.append(labels.numpy())

    fold_logits = np.concatenate(fold_logits)
    fold_labels = np.concatenate(fold_labels)

    for i, orig_idx in enumerate(val_df.index):
        oof_logits[orig_idx] = fold_logits[i]
        oof_labels[orig_idx] = fold_labels[i]

    auc = roc_auc_score(fold_labels[:, 0], 1/(1+np.exp(-fold_logits[:, 0])))
    print(f'  Fold {fold_id}: {len(val_df):,} samples, AUC_any={auc:.5f}')

    del model
    torch.cuda.empty_cache()
    gc.collect()

oof_probs = 1 / (1 + np.exp(-oof_logits))
global_auc = roc_auc_score(oof_labels[:, 0], oof_probs[:, 0])
print(f'\nOOF AUC_any = {global_auc:.5f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  EXPECTED CALIBRATION ERROR (ECE)  ██
# ══════════════════════════════════════════════════════════════════════════

def compute_ece(y_true, y_prob, n_bins=15):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_data = []
    for b in range(n_bins):
        lo, hi = bin_boundaries[b], bin_boundaries[b+1]
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            bin_data.append((0, 0, 0))
            continue
        avg_conf = y_prob[mask].mean()
        avg_acc  = y_true[mask].mean()
        w = mask.sum() / len(y_true)
        ece += w * abs(avg_acc - avg_conf)
        bin_data.append((avg_conf, avg_acc, mask.sum()))
    return ece, bin_data


def reliability_diagram(y_true, y_prob, title, ax, n_bins=15):
    ece, bin_data = compute_ece(y_true, y_prob, n_bins)
    confs = [b[0] for b in bin_data if b[2] > 0]
    accs  = [b[1] for b in bin_data if b[2] > 0]
    ax.bar(confs, accs, width=1/n_bins*0.8, alpha=0.6, color='steelblue', edgecolor='k')
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
    ax.set(xlabel='Mean predicted prob', ylabel='Fraction positive', title=f'{title}\nECE = {ece:.4f}')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend()


# Before calibration
ece_before, _ = compute_ece(oof_labels[:, 0], oof_probs[:, 0])
brier_before  = brier_score_loss(oof_labels[:, 0], oof_probs[:, 0])
print(f'BEFORE calibration:  ECE = {ece_before:.4f},  Brier = {brier_before:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  TEMPERATURE SCALING  ██
# ══════════════════════════════════════════════════════════════════════════

def nll_with_temp(T, logits, labels):
    """Negative log-likelihood for temperature T."""
    scaled = logits / T
    probs = 1 / (1 + np.exp(-scaled))
    probs = np.clip(probs, 1e-7, 1 - 1e-7)
    return -(labels * np.log(probs) + (1 - labels) * np.log(1 - probs)).mean()

# Optimize temperature for 'any' class
result = minimize_scalar(
    lambda T: nll_with_temp(T, oof_logits[:, 0], oof_labels[:, 0]),
    bounds=(0.1, 10.0), method='bounded')
best_T = result.x

ts_probs = 1 / (1 + np.exp(-oof_logits[:, 0] / best_T))
ece_ts, _ = compute_ece(oof_labels[:, 0], ts_probs)
brier_ts  = brier_score_loss(oof_labels[:, 0], ts_probs)

print(f'Optimal temperature: {best_T:.4f}')
print(f'After temp scaling:  ECE = {ece_ts:.4f} (was {ece_before:.4f}),')
print(f'                     Brier = {brier_ts:.4f} (was {brier_before:.4f})')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  ISOTONIC REGRESSION  ██
# ══════════════════════════════════════════════════════════════════════════

iso_models = {}
for i, name in enumerate(SUBTYPES):
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(oof_probs[:, i], oof_labels[:, i])
    iso_models[name] = iso

iso_probs_any = iso_models['any'].predict(oof_probs[:, 0])
ece_iso, _ = compute_ece(oof_labels[:, 0], iso_probs_any)
brier_iso = brier_score_loss(oof_labels[:, 0], iso_probs_any)

print(f'After isotonic reg:  ECE = {ece_iso:.4f}, Brier = {brier_iso:.4f}')
print(f'\nCalibration comparison for "any" class:')
print(f'  Raw       : ECE={ece_before:.4f}  Brier={brier_before:.4f}')
print(f'  Temp scale: ECE={ece_ts:.4f}  Brier={brier_ts:.4f}  (T={best_T:.3f})')
print(f'  Isotonic  : ECE={ece_iso:.4f}  Brier={brier_iso:.4f}')

# Best method
best_method = 'isotonic' if ece_iso < ece_ts else 'temperature'
print(f'\n→ Best calibration method: {best_method}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  RELIABILITY DIAGRAMS  ██
# ══════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
reliability_diagram(oof_labels[:, 0], oof_probs[:, 0], 'Raw', axes[0])
reliability_diagram(oof_labels[:, 0], ts_probs,         f'Temp scaling (T={best_T:.2f})', axes[1])
reliability_diagram(oof_labels[:, 0], iso_probs_any,    'Isotonic regression', axes[2])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'reliability_diagrams.png', dpi=150, bbox_inches='tight')
plt.show()
print('Reliability diagrams saved.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  3-BAND TRIAGE SYSTEM + CLINICAL OPERATING POINTS  ██
# ══════════════════════════════════════════════════════════════════════════

# Use the best calibrated probabilities
if best_method == 'isotonic':
    cal_probs = iso_probs_any
else:
    cal_probs = ts_probs

# Triage thresholds
# REVIEWER NOTE: These thresholds are HEURISTIC values, not optimized on
# this dataset. They are informed by general clinical triage principles:
#   HIGH ≥ 0.7  → targets high specificity for immediate review
#   LOW  < 0.3  → targets high sensitivity (miss <5%) for safe deferral
# In clinical deployment, these would require institutional calibration
# against local prevalence, staffing, and acceptable miss rates.
# We report them to demonstrate the triage concept, not as validated cutoffs.
HIGH_THRESH = 0.7
LOW_THRESH  = 0.3

triage = np.where(cal_probs >= HIGH_THRESH, 'HIGH',
         np.where(cal_probs >= LOW_THRESH,  'MEDIUM', 'LOW'))

gt_any = oof_labels[:, 0].astype(int)

print('3-Band Triage Summary')
print('=' * 50)
for band in ['HIGH', 'MEDIUM', 'LOW']:
    mask = triage == band
    n = mask.sum()
    pos = gt_any[mask].sum()
    rate = pos / n if n > 0 else 0
    print(f'  {band:6s}  n={n:6,}  positives={pos:5,}  prevalence={rate:.3f}')

# Safety: missed hemorrhages in LOW band
low_mask = triage == 'LOW'
missed = gt_any[low_mask].sum()
total_pos = gt_any.sum()
miss_rate = missed / total_pos if total_pos > 0 else 0
print(f'\n  Missed hemorrhages in LOW band: {missed}/{int(total_pos)} ({miss_rate:.3%})')
print(f'  Safety criterion (miss <5%): {"✅ PASS" if miss_rate < 0.05 else "❌ FAIL"}')

# ─── Clinical Operating Points ──────────────────────────────────────
fpr_cal, tpr_cal, thr_cal = roc_curve(gt_any, cal_probs)
spec_cal = 1 - fpr_cal

# Sensitivity @ 90% specificity
idx_spec90 = np.argmin(np.abs(spec_cal - 0.90))
sens_at_spec90 = tpr_cal[idx_spec90]
thr_at_spec90  = thr_cal[min(idx_spec90, len(thr_cal)-1)]

# Specificity @ 95% sensitivity
idx_sens95 = np.argmin(np.abs(tpr_cal - 0.95))
spec_at_sens95 = spec_cal[idx_sens95]
thr_at_sens95  = thr_cal[min(idx_sens95, len(thr_cal)-1)]

print(f'\n{"="*50}')
print(f'  CLINICAL OPERATING POINTS (calibrated probs)')
print(f'{"="*50}')
print(f'  Sensitivity @ 90% Specificity: {sens_at_spec90:.4f}  (threshold={thr_at_spec90:.4f})')
print(f'  Specificity @ 95% Sensitivity: {spec_at_sens95:.4f}  (threshold={thr_at_sens95:.4f})')

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
bands = ['HIGH', 'MEDIUM', 'LOW']
colors_b = ['#e53935', '#ff9800', '#4caf50']
for i, band in enumerate(bands):
    m = triage == band
    pos = gt_any[m].sum()
    neg = m.sum() - pos
    ax.barh(band, pos, color=colors_b[i], alpha=0.8, label='Positive' if i == 0 else '')
    ax.barh(band, neg, left=pos, color=colors_b[i], alpha=0.3, label='Negative' if i == 0 else '')
    ax.text(m.sum() + 50, i, f'{m.sum():,}', va='center')

ax.set_xlabel('Number of scans')
ax.set_title('Triage Band Distribution')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'triage_bands.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  SAVE CALIBRATION PARAMS  ██
# ══════════════════════════════════════════════════════════════════════════
import pickle

calib_params = {
    'best_method': best_method,
    'temperature': float(best_T),
    'ece_raw': float(ece_before),
    'ece_temp': float(ece_ts),
    'ece_isotonic': float(ece_iso),
    'brier_raw': float(brier_before),
    'brier_temp': float(brier_ts),
    'brier_isotonic': float(brier_iso),
    'triage_high_thresh': HIGH_THRESH,
    'triage_low_thresh': LOW_THRESH,
    'oof_auc_any': float(global_auc),
    'sensitivity_at_spec90': float(sens_at_spec90),
    'specificity_at_sens95': float(spec_at_sens95),
    'threshold_at_spec90': float(thr_at_spec90),
    'threshold_at_sens95': float(thr_at_sens95),
}
json.dump(calib_params, open(OUTPUT_DIR / 'calibration_params.json', 'w'), indent=2)

# Save isotonic models
with open(OUTPUT_DIR / 'isotonic_models.pkl', 'wb') as f:
    pickle.dump(iso_models, f)

# Save OOF predictions for NB05
oof_df = manifest[['image_id', 'patient_id', 'fold'] + SUBTYPES].copy()
for i, name in enumerate(SUBTYPES):
    oof_df[f'prob_{name}'] = oof_probs[:, i]
    oof_df[f'logit_{name}'] = oof_logits[:, i]
oof_df.to_csv(OUTPUT_DIR / 'oof_predictions.csv', index=False)

print('Saved:')
print('  calibration_params.json (incl. clinical operating points)')
print('  isotonic_models.pkl')
print('  oof_predictions.csv')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ██  HEALTH CHECK  ██
# ══════════════════════════════════════════════════════════════════════════
errors = []
for f in ['calibration_params.json', 'isotonic_models.pkl',
          'oof_predictions.csv', 'reliability_diagrams.png', 'triage_bands.png']:
    if not (OUTPUT_DIR / f).exists():
        errors.append(f'{f} missing')
if ece_before < 0 or ece_before > 1:
    errors.append(f'ECE out of range: {ece_before}')

health = {
    'notebook': '04_calibration',
    'status': 'PASS' if not errors else 'FAIL',
    'errors': errors,
    'ece_raw': round(ece_before, 5),
    'ece_best': round(min(ece_ts, ece_iso), 5),
    'best_method': best_method,
    'oof_auc_any': round(global_auc, 5),
}
json.dump(health, open(OUTPUT_DIR / 'health_check_nb04.json', 'w'), indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:', errors)
else:
    print('✅ HEALTH CHECK PASSED')
    print(f'   OOF AUC_any = {global_auc:.5f}')
    print(f'   ECE: {ece_before:.4f} → {min(ece_ts, ece_iso):.4f} ({best_method})')